# 05. Valutazione e calibrazione

Un decision model restituisce probabilita, quindi l'accuratezza da sola non basta. Ci chiediamo anche se le probabilita sono calibrate: quando il modello dice 0.8, dovrebbe avere ragione circa l'80 percento delle volte?

Calcoliamo quattro cose: accuratezza, Brier score, errore di calibrazione atteso e una soglia di confidenza che passa oltre le decisioni incerte. Tutte le celle sono piccole e terminano con una validazione.


## Passo 1. Caricare il dataset e un client

Usiamo il mock cosi il notebook e riproducibile senza server. In seguito sostituite con un client reale; il codice di valutazione non cambia.


In [ ]:
from decision_qa import MockClient, read_jsonl
records = read_jsonl('data/pubmedqa_decisions.jsonl')
print(len(records))


In [ ]:
client = MockClient(seed=42, quality=0.75)
print(type(client).__name__)


## Passo 2. Un risultato, una probabilita

`gold_probability` estrae la probabilita che il modello ha assegnato alla risposta corretta. Lo mostriamo su una singola domanda prima di iterare.


In [ ]:
from decision_qa.evaluate import gold_probability
record = records[0]
question = record.questions[0]
result = client.decision(record.state, [question])[0]
print(result.probabilities)


In [ ]:
p = gold_probability(result, question)
print(round(p, 3))
assert 0.0 <= p <= 1.0


## Passo 3. Raccogliere tutte le previsioni

Scorriamo ogni record e ogni domanda, memorizzando la probabilita gold e un flag binario che indica se la risposta corretta era quella con probabilita massima.


In [ ]:
gold_probs = []
correct = []
for rec in records:
    for q, res in zip(rec.questions, client.decision(rec.state, rec.questions)):
        gp = gold_probability(res, q)
        gold_probs.append(gp)
        correct.append(int(gp >= max(res.probabilities.values()) - 1e-12))


In [ ]:
print(len(gold_probs), len(correct))
assert len(gold_probs) == len(correct)


## Passo 4. Accuratezza

L'accuratezza e la frazione di domande in cui la risposta corretta era quella con probabilita massima. Non dice nulla sulla qualita dei valori di probabilita.


In [ ]:
acc = sum(correct) / len(correct)
print(round(acc, 3))
assert 0.0 <= acc <= 1.0


## Passo 5. Brier score

Il Brier score e l'errore quadratico medio delle probabilita gold. Piu basso e meglio. Un modello accurato e sicuro di se si avvicina a zero.


In [ ]:
from decision_qa import brier
score = brier(gold_probs)
print(round(score, 3))
assert 0.0 <= score <= 1.0


In [ ]:
one = brier([0.8])
print(round(one, 3))
assert abs(one - 0.04) < 1e-9


## Passo 6. Affidabilita ed ECE

La tabella di affidabilita raggruppa le previsioni per confidenza e confronta la confidenza media con l'accuratezza osservata. L'ECE e lo scarto pesato tra i due.


In [ ]:
from decision_qa.evaluate import reliability
rows = reliability(gold_probs, correct)
print(len(rows))


In [ ]:
for row in rows:
    print(row['bin'], row['count'], round(row['confidence'], 2), round(row['accuracy'], 2))


In [ ]:
from decision_qa import ece
error = ece(gold_probs, correct)
print(round(error, 3))
assert 0.0 <= error <= 1.0


## Passo 7. Una soglia di confidenza

`fit_threshold` trova il taglio piu basso che raggiunge una precisione target su questi dati. Le decisioni sotto il taglio vanno passate a un modello piu forte. In un progetto reale la soglia va calcolata su uno split di validazione, mai su quello di test.


In [ ]:
from decision_qa import fit_threshold
threshold = fit_threshold(gold_probs, correct, target_precision=0.9)
print(round(threshold, 2))


In [ ]:
coverage = sum(p >= threshold for p in gold_probs) / len(gold_probs)
print(round(coverage, 3))
assert 0.0 <= coverage <= 1.0


## Passo 8. Invarianza all'ordine delle opzioni

Infine verifichiamo che una choice etichettata mantenga la stessa risposta quando le opzioni vengono riordinate. Questo protegge dal bias posizionale.


In [ ]:
from decision_qa.evaluate import option_order_invariance
from decision_qa import ChoiceQuestion
q = ChoiceQuestion(label='check', options=['a', 'b', 'c'], gold_index=1)
invariant = option_order_invariance(client, {}, q)
print(invariant)


In [ ]:
assert invariant is True
print('decision is stable under reordering')


## Leggere i risultati con spirito critico

Alta accuratezza con scarsa calibrazione significa che le classifiche sono utili ma le probabilita non vanno fidate come numeri. Riferimenti gratuiti: JevBench sotto licenza MIT, ProbabilityBench, l'articolo Evaluating and Benchmarking the System One model, l'Early Evidence Audit con la sua checklist a 14 punti e Beyond Calibration. JevAdvBench ricorda che lo stato e input non attendibile, e Jev Does Not Play Dice mostra quanto le probabilita possano allontanarsi dalla realta.
